# 공통 분석 파이프라인 — 수치 탐색에서 해석적 증명까지

이 발표의 중심 질문은 다음과 같다. **SDP의 계수 수치해를 해석적인 potential inequality와 수렴률 증명으로 어떻게 연결할 것인가?** GD에서 공통 분석 절차를 설명한 뒤, SGD에서 확률적 오차를 반영하고, averaging에서 최종 상한의 오차항을 줄이는 방법을 살펴본다.

발표는 다음 순서로 진행한다.

1. **공통 분석 파이프라인:** 계수 탐색 문제가 LMI와 SDP가 되는 이유
2. [**Example I · GD**](3_2_1_Figure_1_1A_GradientDescent.ipynb): Figure 1의 수치 탐색에서 Theorem 3까지
3. [**Example II · SGD**](3_2_2_Figure_2_2A_StochasticGradientDescent.ipynb): 같은 분석법에 분산을 반영하고 Figure 2와 Theorem 5로 연결
4. [**Averaging 확장**](Averaging_expansion_Theorem_6-7_Table_2.ipynb): 평균점과 oracle 평가점을 바꾸어 Theorem 6·7을 비교하고 Table 2로 종합

세 노트북의 분석을 다음 다섯 단계로 정리한다.

| 단계 | 수학적 과제 | 결과 |
| --- | --- | --- |
| **1. 알고리즘과 가정** | 업데이트식, 함수 클래스, oracle 가정 설정 | 한 단계 갱신식 및 함수·oracle 제약 |
| **2. Parameterized potential 설정** | $\phi_k^f$의 구조 및 $\phi_0^f,\phi_N^f$의 경계조건 | 계수 변수와 한 단계 potential inequality |
| **3. LMI/SDP로 계수 수치 탐색** | Gram 표현, interpolation 부등식, 분산 제약의 LMI화 | 함수값 계수 소거 조건, LMI, SDP 수치해 |
| **4. 계수 패턴 관찰과 단순화** | 수치 계수의 $k$-의존성과 추가 제약의 영향 분석 | 명시적 계수 수열의 후보 |
| **5. 해석적 potential inequality와 수렴률** | multiplier·계수 점화식의 명시적 구성 | 한 단계 부등식의 증명과 수렴률 |

GD와 SGD 노트북에서는 각 알고리즘을 위의 다섯 단계에 따라 분석한다. Averaging 노트북에서는 두 평균법을 같은 절차로 각각 분석한 뒤, 함수값 상한과 수렴률을 비교한다.

## 1단계 · 알고리즘과 가정

$f\in\mathcal F_L$, $x_\star\in\arg\min f$, $f_\star=f(x_\star)$로 둔다. 각 사례의 업데이트식은 다음과 같다. GD에서는

$$x_{k+1}=x_k-\frac1Lf'(x_k)$$

를 사용하고, SGD에서는 확률적 gradient를 반환하는 oracle을 사용한다.

$$x_{k+1}^{(i)}=x_k-\delta_kG(x_k;i),\qquad
\mathbb E_iG(x_k;i)=f'(x_k),\qquad
\mathbb E_i\|G(x_k;i)-f'(x_k)\|^2\le\sigma^2.$$

$\mathbb E_i$는 현재 상태까지의 정보를 고정한 조건부 기대값이다. 수치 실험에서는 oracle이 유한 개의 값 중 하나를 반환하도록 설정한다. 각 SDP를 풀 때 step-size와 업데이트 식은 고정되어 있다.

| 사례 | 앞 사례에서 이어받는 것 | 새로 달라지는 것 |
| --- | --- | --- |
| GD | 공통 분석법을 처음 구성 | 정확한 gradient를 사용하는 업데이트 |
| SGD | 함수 클래스와 potential·Gram 표현·interpolation 부등식·LMI 구성 | 조건부 기대값과 분산 상한 |
| SGD with averaging | SGD의 함수·oracle 가정과 업데이트 | SGD 갱신 후 반복점들의 평균 $z_k$에서 함수값을 평가 |
| Primal averaging | 같은 함수·oracle 가정과 평균점의 함수값을 분석하는 목표 | 평균점 $y_{k+1}$을 먼저 정하고 그 점에서 oracle을 평가 |

따라서 averaging으로 넘어갈 때도 동일한 분석법을 사용한다. 바뀐 업데이트가 한 단계 부등식과 오차 계수에 어떤 차이를 만드는지가 새 질문이다.

## 2단계 · Parameterized potential 설정

GD와 SGD에서는 거리 제곱, gradient norm 제곱, 교차항, 함수값으로 이루어진 potential을 사용한다.

$$\phi_k^f=
a_k\|x_k-x_\star\|^2+b_k\|f'(x_k)\|^2
+2c_k\langle f'(x_k),x_k-x_\star\rangle
+d_k(f(x_k)-f_\star).$$

평균법에서는 함수값과 gradient 관련 항을 $z_k$ 또는 $y_k$에서 평가하고, SGD 반복점 $x_k$의 거리항을 별도로 둔다. 알고리즘과 경계조건으로 고정한 계수 외에, 나머지 계수들을 한 단계의 potential 부등식을 만족하도록 찾는다.

$$\boxed{\mathbb E_i\phi_{k+1}^f\le\phi_k^f+e_k\sigma^2.}$$

GD에서는 기대값을 생략하고 $e_k=0$으로 둔다. 경계조건 $\phi_0^f,\phi_N^f$는 분석하려는 성능지표에 맞춰 정한다.

| 사례 | 대표적인 $\phi_N^f$ | 유도할 상한 |
| --- | --- | --- |
| GD (기본 SDP) | $b_N\|f'(x_N)\|^2$ | $\|f'(x_N)\|^2$ |
| SGD (기본 SDP) | $d_N(f(x_N)-f_\star)$ | $\mathbb E[f(x_N)-f_\star]$ |
| SGD with averaging (Theorem 6) | $D_N(f(z_N)-f_\star)+\frac L2\|x_N-x_\star\|^2$ | $\mathbb E[f(z_N)-f_\star]$ |
| Primal averaging (Theorem 7) | $d_N(f(y_N)-f_\star)+\frac L2\|x_N-x_\star\|^2$ | $\mathbb E[f(y_N)-f_\star]$ |

표에는 기본 GD·SGD SDP와 Theorem 6·7의 경계조건을 제시했다. SDP의 계수 제한(`relax`)에 따라 $\phi_N^f$의 거리항 유무는 달라진다. Theorem 6의 $D_k$는 논문 Appendix E.5의 계수로, $D_0=0$, $D_k=L\delta_{k-1}(d_{k-1}+1)$이다. Theorem 7은 논문의 $d_k$를 그대로 사용한다. 각 사례에서 계수 수열의 정의를 따로 적용한다.

이제 potential의 형태와 경계조건을 정했으므로, 한 단계 부등식을 LMI로 나타낸다.

## 3단계 · LMI/SDP로 계수 수치 탐색

### Gram 표현 $G=P^\top P$

논문 Appendix D.4, pp.35–36의 표기를 사용한다. 한 단계에서 필요한 실제 벡터를 $P$의 열로 모은다. 일반성을 잃지 않고 $x_\star=f_\star=0$으로 두면 GD에서는

$$P=[x_k\quad f'(x_k)\quad f'(x_{k+1})],\qquad
F=[f(x_k)\quad f(x_{k+1})],\qquad G=P^\top P\succeq0.$$

$G$의 성분은 $P$의 열 사이의 내적이다. $F$는 논문과 같은 함수값 **행벡터**다. 점·gradient·함수값을 고르는 논문의 계수벡터는 다음과 같다.

$$\begin{aligned}
\mathbf x_k&=(1,0,0)^\top,&\mathbf g_k&=(0,1,0)^\top,&\mathbf g_{k+1}&=(0,0,1)^\top,\\
\mathbf x_{k+1}&=(1,-L^{-1},0)^\top,&\mathbf x_\star&=0,&\mathbf g_\star&=0,\\
\mathbf f_k&=(1,0)^\top,&\mathbf f_{k+1}&=(0,1)^\top,&\mathbf f_\star&=0.
\end{aligned}$$

즉 $P\mathbf x_p=x_p$, $P\mathbf g_p=f'(x_p)$, $F\mathbf f_p=f(x_p)$이다. SGD와 평균법에서는 각 알고리즘에 필요한 oracle 결과와 다음 점의 gradient 등을 $P$에 추가한다. $G$는 Gram 행렬을, $G(x;i)$는 stochastic oracle을 뜻한다.

### Potential difference의 선형성: GD의 명시적 전개

2단계에서 정한 potential을 위의 $G,F$로 쓰면

$$\phi_k^f=a_kG_{11}+b_kG_{22}+2c_kG_{12}+d_kF\mathbf f_k.$$

GD 업데이트 $x_{k+1}=x_k-f'(x_k)/L$를 대입하면

$$\begin{aligned}
\|x_{k+1}-x_\star\|^2
&=G_{11}-\frac2LG_{12}+\frac1{L^2}G_{22},\\
\|f'(x_{k+1})\|^2&=G_{33},\\
\langle f'(x_{k+1}),x_{k+1}-x_\star\rangle
&=G_{13}-\frac1LG_{23}.
\end{aligned}$$

따라서 다음 potential은

$$\begin{aligned}
\phi_{k+1}^f={}&a_{k+1}\left(G_{11}-\frac2LG_{12}+\frac1{L^2}G_{22}\right)
+b_{k+1}G_{33}\\
&+2c_{k+1}\left(G_{13}-\frac1LG_{23}\right)+d_{k+1}F\mathbf f_{k+1}.
\end{aligned}$$

논문 Appendix D.4, p.36에서는 이차항의 계수행렬을 다음과 같이 쓴다.

$$\begin{aligned}
V_k&=[\mathbf x_k-\mathbf x_\star\quad\mathbf g_k]
\begin{pmatrix}a_k&c_k\\c_k&b_k\end{pmatrix}
[\mathbf x_k-\mathbf x_\star\quad\mathbf g_k]^\top,\\
V_{k+1}&=[\mathbf x_{k+1}-\mathbf x_\star\quad\mathbf g_{k+1}]
\begin{pmatrix}a_{k+1}&c_{k+1}\\c_{k+1}&b_{k+1}\end{pmatrix}
[\mathbf x_{k+1}-\mathbf x_\star\quad\mathbf g_{k+1}]^\top.
\end{aligned}$$

현재 potential을 빼면

$$\boxed{\phi_{k+1}^f-\phi_k^f
=\operatorname{Tr}\bigl(G(V_{k+1}-V_k)\bigr)
+F\bigl[d_{k+1}(\mathbf f_{k+1}-\mathbf f_\star)-d_k(\mathbf f_k-\mathbf f_\star)\bigr],}$$

$$V_{k+1}-V_k=\begin{pmatrix}
a_{k+1}-a_k&-a_{k+1}/L-c_k&c_{k+1}\\
-a_{k+1}/L-c_k&a_{k+1}/L^2-b_k&-c_{k+1}/L\\
c_{k+1}&-c_{k+1}/L&b_{k+1}
\end{pmatrix}.$$

Potential 계수를 고정하면 $\phi_{k+1}^f-\phi_k^f$는 Gram 성분과 함수값에 대한 선형식이다. 또한 $1/L$가 고정되어 있으므로 $V_{k+1}-V_k$의 각 성분도 potential 계수의 선형식이다. 예를 들어 $(V_{k+1}-V_k)_{12}=-a_{k+1}/L-c_k$에는 미지 계수 사이의 곱이 없다. 원본 구현에서는 이 두 이차항을 각각 `statesK1G(k) @ Q[k+1] @ statesK1G(k).T`와 `statesKG @ Q[k] @ statesKG.T`로 조립한다. 이 코드 식의 `k`는 원본과 같이 1부터 시작한다.

### Interpolation 부등식과 함수값 계수 소거

$\mathcal F_L$에 대한 **smooth convex interpolation inequality**는 다음과 같다. 여러 점의 함수값과 gradient가 동일한 $f\in\mathcal F_L$에서 얻어질 수 있는 조건이다.

$$f(x_p)-f(x_q)-\langle f'(x_q),x_p-x_q\rangle
-\frac1{2L}\|f'(x_p)-f'(x_q)\|^2\ge0.$$

GD에서는 논문의 $I_k=\{k,k+1,\star\}$를 사용한다. Appendix D.4의 식 (19), p.35에 따라

$$\begin{aligned}
M&=\frac12\begin{pmatrix}
0&0&0&-1\\0&0&0&1\\0&0&-L^{-1}&L^{-1}\\-1&1&L^{-1}&-L^{-1}
\end{pmatrix},\\
M_{p,q}&=[\mathbf x_p\ \mathbf x_q\ \mathbf g_p\ \mathbf g_q]
M[\mathbf x_p\ \mathbf x_q\ \mathbf g_p\ \mathbf g_q]^\top
\end{aligned}$$

로 쓰면 interpolation 부등식은 $F(\mathbf f_p-\mathbf f_q)+\operatorname{Tr}(GM_{p,q})\ge0$이다. 예를 들어 $p=\star,q=k$이면

$$-F\mathbf f_k+G_{12}-\frac1{2L}G_{22}\ge0,\qquad
M_{\star,k}=\begin{pmatrix}0&1/2&0\\1/2&-1/(2L)&0\\0&0&0\end{pmatrix}.$$

$L$과 업데이트를 고정하면 모든 $M_{p,q}$는 상수행렬이다. 각 순서쌍에 $\lambda_{p,q}\ge0$를 부여하고 potential 차이에 더하면

$$\begin{aligned}
&\phi_{k+1}^f-\phi_k^f
+\sum_{p,q\in I_k}\lambda_{p,q}\bigl[F(\mathbf f_p-\mathbf f_q)+\operatorname{Tr}(GM_{p,q})\bigr]\\
&\quad=\operatorname{Tr}\!\left[G\left(V_{k+1}-V_k+\sum_{p,q\in I_k}\lambda_{p,q}M_{p,q}\right)\right]\\
&\qquad+F\begin{pmatrix}
-d_k-\lambda_{\star,k}+\lambda_{k,\star}+\lambda_{k,k+1}-\lambda_{k+1,k}\\
d_{k+1}-\lambda_{\star,k+1}-\lambda_{k,k+1}+\lambda_{k+1,\star}+\lambda_{k+1,k}
\end{pmatrix}.
\end{aligned}$$

이 가중합이 모든 $G\succeq0,F\in\mathbb R^{1\times2}$에서 0 이하가 되도록 요구한다. 함수값 항의 계수가 0이어야 하고, 남은 trace 항은 다음 행렬 조건으로 0 이하가 된다. 논문 p.36 하단 박스의 조건을 성분으로 풀어 쓰면

$$\boxed{\begin{gathered}
-d_k-\lambda_{\star,k}+\lambda_{k,\star}+\lambda_{k,k+1}-\lambda_{k+1,k}=0,\\
d_{k+1}-\lambda_{\star,k+1}-\lambda_{k,k+1}+\lambda_{k+1,\star}+\lambda_{k+1,k}=0,\\
V_{k+1}-V_k+\sum_{p,q\in I_k}\lambda_{p,q}M_{p,q}\preceq0,\qquad\lambda_{p,q}\ge0.
\end{gathered}}$$

원래의 interpolation 부등식을 만족하는 점들에서는 더한 가중합이 비음수다. 따라서 위 조건들로 $\phi_{k+1}^f-\phi_k^f\le0$을 보장한다. $p=q$인 항은 0이므로 원본 구현은 여섯 개의 서로 다른 순서쌍만 더한다.

행렬의 성분도 명시적으로 보면

$$\begin{aligned}
\left(V_{k+1}-V_k+\sum_{p,q\in I_k}\lambda_{p,q}M_{p,q}\right)_{12}
&=-\frac{a_{k+1}}L-c_k+\frac12\lambda_{\star,k},\\
\left(V_{k+1}-V_k+\sum_{p,q\in I_k}\lambda_{p,q}M_{p,q}\right)_{22}
&=\frac{a_{k+1}}{L^2}-b_k
+\frac{\lambda_{k+1,k}-\lambda_{\star,k}-\lambda_{k,\star}-\lambda_{k,k+1}}{2L}.
\end{aligned}$$

$L$이 고정되어 있으므로 각 성분은 potential 계수와 multiplier의 선형식이다. 일부 계수를 경계조건으로 고정하면 상수항이 생길 수 있으므로 일반적으로 결정변수에 대해 **affine**하다고 말한다. 따라서 위 음의 준정부호 조건이 **Linear Matrix Inequality (LMI)**다. 원본 코드에서는 행렬의 좌변을 `cons_SDP[k]`, 함수값 계수의 행벡터를 `cons_LIN[k]`에 저장한다.

### 확률적 방법: 분산 제약 추가

SGD의 논문 Appendix E.7, pp.48–49에서는 현재 점의 계수벡터를 $\mathbf x_0$, outcome $i$에 따른 다음 점의 계수벡터를 $\mathbf x_i$로 쓴다. $P\mathbf x_0=x_k$, $P\mathbf x_i=x_{k+1}^{(i)}$, $P\mathbf g_0=f'(x_k)$, $P\mathbf g_i=f'(x_{k+1}^{(i)})$, $P\mathbf G_i=G(x_k;i)$이다. 함수값 계수벡터도 $F\mathbf f_0=f(x_k)$, $F\mathbf f_i=f(x_{k+1}^{(i)})$로 정한다. 각 outcome은 $1/n$의 확률로 선택된다. Unbiasedness와 업데이트는 논문 p.49의 계수벡터 관계

$$\mathbf g_0=\frac1n\sum_{i=1}^n\mathbf G_i,\qquad
\mathbf x_i=\mathbf x_0-\delta_k\mathbf G_i$$

로 반영한다. 해당 Gram 좌표에서 논문의 행렬은

$$\begin{aligned}
V_k&=[\mathbf x_0-\mathbf x_\star\quad\mathbf g_0]
\begin{pmatrix}a_k&c_k\\c_k&b_k\end{pmatrix}
[\mathbf x_0-\mathbf x_\star\quad\mathbf g_0]^\top,\\
V_{k+1}^{(i)}&=[\mathbf x_i-\mathbf x_\star\quad\mathbf g_i]
\begin{pmatrix}a_{k+1}&c_{k+1}\\c_{k+1}&b_{k+1}\end{pmatrix}
[\mathbf x_i-\mathbf x_\star\quad\mathbf g_i]^\top,\\
A_{\rm var}&=\frac1n\sum_{i=1}^n(\mathbf G_i-\mathbf g_0)(\mathbf G_i-\mathbf g_0)^\top,
\qquad \operatorname{Tr}(GA_{\rm var})\le\sigma^2.
\end{aligned}$$

원본 구현의 `Avar`가 이 분산 행렬에 해당한다. 보이고 싶은 $\mathbb E_i\phi_{k+1}^f-\phi_k^f-e_k\sigma^2\le0$의 좌변에 interpolation 부등식의 가중합과 $e_k(\sigma^2-\operatorname{Tr}(GA_{\rm var}))$를 더한다. $-e_k\sigma^2$와 $+e_k\sigma^2$가 소거되므로 행렬에는 $-e_kA_{\rm var}$가 남는다. 논문의 조건은

$$\boxed{\begin{gathered}
\frac1n\sum_{i=1}^nV_{k+1}^{(i)}-V_k-e_kA_{\rm var}
+\sum_{p,q\in I}\lambda_{p,q}M_{p,q}\preceq0,\\
\frac{d_{k+1}}n\sum_{i=1}^n\mathbf f_i-d_k\mathbf f_0
+\sum_{p,q\in I}\lambda_{p,q}(\mathbf f_p-\mathbf f_q)=0,\\
\lambda_{p,q}\ge0,\qquad e_k\ge0,\qquad I=\{\star,0,1,\ldots,n\}.
\end{gathered}}$$

각 SDP에서 업데이트를 고정하면 $V_k,V_{k+1}^{(i)}$의 성분은 potential 계수의 선형식이고, $M_{p,q},A_{\rm var}$는 상수행렬이다. 따라서 위 행렬의 성분은 potential 계수, $\lambda_{p,q},e_k$에 대해 affine하다.

평균법도 해당 알고리즘의 상태와 potential을 Gram 표현으로 전개해 같은 방식으로 구성한다. 원본 B·C 코드에서는 이를 `cons_SDP[k]`, 함수값 계수를 `cons_LIN[k]`에 직접 조립한다. 각 반복 단계의 선형 등식과 LMI, 부호 조건을 모으고 선형 목적함수를 정하면 **SDP**가 된다. 여기서 최적화하는 것은 potential의 계수와 multiplier다.

### SDP 목적함수

목적함수는 2단계에서 정한 최종 상한에 맞춘다. 아래 표의 계수는 원본 A·B·C 코드의 `d[k]`, `e[k]`에 대응한다. B 코드의 함수값 계수 $d_k$는 Theorem 6의 평균 가중치 $d_k$와 역할이 다르며, 코드 설명과 정리의 증명에서 각각 원래 정의를 사용한다.

| 사례 | SDP의 목적함수 | 최종 상한에서의 의미 |
| --- | --- | --- |
| GD | $b_N$ 최대화 | gradient norm 제곱 상한의 분모를 키움 |
| SGD | 먼저 $d_N$ 최대화, 그 수준을 거의 유지하며 $\sum e_k$ 최소화 | 초기 거리 항과 분산 항에 곱해지는 계수를 차례로 줄임 |
| SGD with averaging | 먼저 $d_N$ 최대화, 그 수준을 거의 유지하며 $\sum e_k$ 최소화 | 평균점의 함수값 상한에서 초기 거리 항과 분산 항의 계수를 차례로 줄임 |
| Primal averaging | 정해진 $d_k$ 아래에서 $\sum e_k$ 최소화 | $d_{k+1}=d_k+L\delta_k$가 평균 업데이트에 이미 들어감 |

이처럼 목적함수는 각 사례에서 구하려는 상한에 따라 정한다. Primal averaging에서는 함수값 계수가 알고리즘으로 정해지므로, 단계별 오차 계수의 합을 최소화하는 SDP를 한 번 푼다.

SDP로 얻은 $\{a_k,b_k,c_k,d_k,e_k\}$ 등의 수치 계수 수열에서 $k$에 따른 패턴을 분석한다.

## 4단계 · 계수 패턴 관찰과 단순화

수치해에서 계수의 변화 양상을 살펴보고, potential을 단순화할 조건을 정한다. 그 조건을 적용한 SDP를 다시 풀어, 최종 상한이 얼마나 달라지는지 비교한다.

| 사례 | 계수 수열의 비교 항목 | 단순화한 $\phi_k^f$ |
| --- | --- | --- |
| GD | 최종 $b_N$의 증가, 함수값 계수의 선형 형태, 교차항·거리항의 계수를 제한했을 때의 변화 | 거리항 계수를 고정하고 교차항을 없앤 potential |
| SGD | $d_N$과 오차 계수의 누적합 사이의 관계, 교차항·gradient norm 제곱 항을 없애고 거리항을 고정했을 때의 변화 | $\phi_k^f=(L/2)\|x_k-x_\star\|^2+d_k(f(x_k)-f_\star)$ |
| 두 평균법 | 평균점의 이차항을 없앤 경우, 최종 거리항의 선택, 함수값 계수와 단계별 오차 계수의 관계 | 평균점의 함수값과 $x_k$의 거리항으로 이루어진 potential |

계수를 제한하는 실험에는 초기·최종 조건이 함께 바뀌는 경우도 있다. 각 노트북은 그 차이를 밝힌 상태에서 수치해를 비교한다. 확률적 방법에서는 함수값 계수뿐 아니라 **오차 계수의 누적합을 함수값 계수로 나눈 비율**까지 비교한다.

다음 단계에서는 계수 점화식과 multiplier를 명시적으로 구성해 $\mathbb E_i\phi_{k+1}^f\le\phi_k^f+e_k\sigma^2$를 증명한다.

## 5단계 · 해석적 potential inequality와 수렴률

단순화한 potential의 계수를 점화식이나 명시적 수열로 정하고, 사용할 interpolation 부등식과 비음수 multiplier를 선택한다. 가중합을 전개했을 때 함수값 항이 소거되거나 potential 차이로 모이고, 남은 제곱항의 부호가 정해지는 이유를 증명한다.

3단계에서는 조건을 만족하는 계수를 수치적으로 구했다. 여기서는 multiplier를 명시적인 식으로 제시하고, 함수값 항의 소거 조건과 논문의 행렬 부등식이 각 반복 단계에서 성립함을 증명한다.

| 정리 | 5단계에서 증명할 내용 |
| --- | --- |
| Theorem 3 | 세 부등식의 가중합을 GD potential의 차이로 만드는 계수 점화식 |
| Theorem 5 | 조건부 기대값을 취한 SGD potential의 변화량을 분산항으로 제한하는 과정 |
| Theorem 6 | 평균 업데이트 식을 대입하고 완전제곱식으로 정리해 오차 계수를 구하는 과정 |
| Theorem 7 | 평균점에서 oracle을 평가할 때 교차항이 소거되어 더 단순한 오차 계수를 얻는 과정 |

한 단계 부등식을 증명한 뒤에는 기대값을 취하고 합하여

$$\mathbb E\phi_N^f
\le\phi_0^f+\sigma^2\sum_{k=0}^{N-1}e_k$$

를 얻는다. GD에서는 기대값과 오차항을 생략한다. $\phi_N^f$에서 비음수 보조항을 제외해 2단계 표의 성능지표 상한을 얻는다.

마지막으로 $N$에 따른 상한의 차수를 평가해 수렴률을 얻는다.

## 발표 전체를 잇는 질문

사례별 확장은 다음과 같다.

- **GD → SGD:** GD에서 사용한 Gram 표현·interpolation 부등식·LMI 구성에 조건부 기대값과 분산 조건이 들어오면, 새로 어떤 계수가 필요하고 목적함수는 어떻게 달라지는가?
- **SGD → Averaging:** 함수 클래스와 oracle 가정을 유지하면서 평균점을 출력하면, 최종 함수값 상한의 오차항을 얼마나 줄일 수 있는가?
- **Averaging → Primal averaging:** 평균을 만드는 순서와 oracle 평가점을 바꾸면, 같은 분석법으로 구한 오차 계수는 어떻게 달라지는가?
- **두 평균법 → Table 2:** 앞에서 구한 명시적 계수들을 합했을 때, step-size 감소 속도는 초기 거리 항과 분산 항의 균형을 어떻게 바꾸는가?

Table 2 비교에서는 앞에서 구한 계수의 유한 합과 점근적 차수를 분석한다. 따라서 averaging 노트북의 마지막 부분에서는 두 평균법의 분석 결과를 앞선 SGD 결과와 함께 비교한다.

먼저 [**Example I · Gradient descent**](3_2_1_Figure_1_1A_GradientDescent.ipynb)에서 potential 계수 탐색부터 해석적 증명까지 전개한다.

---

참조 범위: 논문 §3.2.1·§3.2.2, Figure 1·2, Theorem 3·5·6·7 및 Table 2. 각 사례 노트북에 해당 수식·참조 코드·저장된 수치 결과와 상세 증명을 배치했다.